In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset

if Run Locally

In [ ]:

cwd = Path.cwd()


if cwd.name == "notebooks" and cwd.parent.name == "Group-Project":
    PROJECT_ROOT = cwd.parent
    os.chdir(PROJECT_ROOT)
    print("Switched cwd 👉", PROJECT_ROOT)
else:
    print(f"当前 cwd = {cwd}，不是位于 '.../Group-Project/notebooks'，跳过切换")

当前 cwd = /content，不是位于 '.../Group-Project/notebooks'，跳过切换


In [ ]:
items_df = pd.read_csv("data/items_cleaned.csv")
items_df.info()

FileNotFoundError: [Errno 2] No such file or directory: 'data/items_cleaned.csv'

If use google colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_path = '/content/drive/MyDrive/CS7643-GroupProject/Data/items_cleaned.csv'
items_df = pd.read_csv(data_path)
items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 402691 entries, 0 to 402690
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   item_Id                 402691 non-null  int64  
 1   ASIN                    402691 non-null  object 
 2   title                   402691 non-null  object 
 3   group                   402691 non-null  object 
 4   similar                 306655 non-null  object 
 5   category_count          402691 non-null  int64  
 6   salesrank_log           402691 non-null  float64
 7   reviews_total_log       402691 non-null  float64
 8   reviews_downloaded_log  402691 non-null  float64
 9   category_path_1         402691 non-null  object 
 10  category_path_2         402691 non-null  object 
 11  category_path_3         402691 non-null  object 
 12  category_path_4         402691 non-null  object 
 13  reviews_avg_ratings     402691 non-null  float64
 14  reviews_avg_votes   

Feature embedding data: file is too large can not uploaded to github.
https://drive.google.com/drive/folders/1LM2YWnPAE-Utr9kLbBBsMeYmxS_wuK6w
data list:
1) title, numercic features are stored in feature_matrix_no_cat.npy / npy.npz
2) title (PCA) , numeric features are stored in feature_matrix_pca_no_cat.npy/npy.npz
3) category feature with option 2 is stored in category_feature_option2.npy
4) Item ID is stored in item_id.npy (could combined later for result interpretability)
5) Category feature embedding option 3 is working in progress.


# Feature embedding

**Model Feature Format Guide**

| Model                             | Feature Usage                   | Preferred Input Format                   | Notes                                                                   |
| --------------------------------- | ------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------- |
| **Node2Vec + MLP**                | Uses features                   | ✅ Dense matrix (`X[i] = [all features]`) | Node2Vec gives structure embedding; you use features for supervised MLP |
| **LightGCN**                      | ❌ No features used              | ❌ Skip feature embedding                 | Uses only structure (item IDs and edges); ignore all features           |
| **GraphSAGE / GAT**               | ✅ Uses features                 | ✅ Dense features required                | Combine all feature blocks into `X: N x D`                              |
| **PinSAGE**                       | ✅ Hybrid (structure + features) | ✅ Dense preferred                        | Node + category/title features; ideal for hybrid recs                   |
| **Temporal Graph Networks (TGN)** | ✅ Time-aware features           | ✅ Dense + timestamps required            | Concatenate dense features; also track time of interaction              |


**Embedding pipeline**:

**Item_df**
| Feature            | Type                        | Embedding Strategy                       | Output Shape per Item |
| ------------------ | --------------------------- | ---------------------------------------- | --------------------- |
| Title         | Text                        | `SentenceTransformer` + **PCA(200)**    | (200,)                |
| Group         | Category (Book, Music…)     | `OneHotEncoder`                          | (11,) or (4,)         |
| Numeric Fields | Numeric (salesrank, counts) | Normalize via `StandardScaler`           | (5,)                  |

**Category_df**
| Feature        | Type                | Embedding Strategy                  | Output Shape                               |
| -------------- | ------------------- | ----------------------------------- | ------------------------------------------ |
| Category paths | Hierarchical string | TF-IDF → **sparse matrix** (or truncatedSVD on dense matrix) |DO NOT RECOMMEND |
| Category paths | Hierarchical string | Label Econded+ Embedding | Starting point
|
| Category paths | Hierarchical string | Graph-based Embedding| IN PROGRESS |



**Review_df**
| Feature                     | Type              | Embedding Strategy                        | Output Shape |
| --------------------------- | ---------------   | ----------------------------------------- | ------------ |
| Ratings, votes, helpfulness | Numeric           | Group by `item_Id`, then `StandardScaler` | (5,)         |





In [ ]:
items_df.shape # cleaned

(402691, 16)

Feature embedding
title → Sentence-BERT (text embedding)

group →  One-hot encoding (categorical)

salesrank, rating, etc. →  normalization

ASIN (node ID) → can be embedded using Node2Vec (graph-based)

In [ ]:
#  Handel missing data
items_df.isnull().sum()

,0
item_Id,0
ASIN,0
title,0
group,0
similar,96036
category_count,0
salesrank_log,0
reviews_total_log,0
reviews_downloaded_log,0
category_path_1,0


In [ ]:
items_df.head(5)

,item_Id,ASIN,title,group,similar,category_count,salesrank_log,reviews_total_log,reviews_downloaded_log,category_path_1,category_path_2,category_path_3,category_path_4,reviews_avg_ratings,reviews_avg_votes,reviews_avg_helpful
0,1,0827229534,Patterns of Preaching: A Sermon Sampler,Book,"0804215715,156101074X,0687023955,0687074231,08...",2,12.891,1.099,1.099,Books[283155],Subjects[1000],Religion & Spirituality[22],Christianity[12290],5.000,8.000,7.000
1,2,0738700797,Candlemas: Feast of Flames,Book,"0738700827,1567184960,1567182836,0738700525,07...",2,12.035,2.565,2.565,Books[283155],Subjects[1000],Religion & Spirituality[22],Earth-Based Religions[12472],4.333,7.000,6.333
2,3,0486287785,World War II Allied Fighter Planes Trading Cards,Book,NaN,1,14.055,0.693,0.693,Books[283155],Subjects[1000],Home & Garden[48],Crafts & Hobbies[5126],5.000,2.000,2.000
3,4,0842328327,Life Application Bible Commentary: 1 and 2 Tim...,Book,0842328610,5,13.356,0.693,0.693,Books[283155],Subjects[1000],Religion & Spirituality[22],Christianity[12290],4.000,1.000,1.000
4,6,0486220125,How the Other Half Lives: Studies Among the Te...,Book,"0486401960,0452283612,0486229076,0714840343",5,12.148,2.890,2.890,Books[283155],Subjects[1000],Arts & Photography[1],Photography[2020],4.235,12.118,8.941


In [ ]:
items_df.tail(10)

,item_Id,ASIN,title,group,similar,category_count,salesrank_log,reviews_total_log,reviews_downloaded_log,category_path_1,category_path_2,category_path_3,category_path_4,reviews_avg_ratings,reviews_avg_votes,reviews_avg_helpful
402681,548542,9627762644,Starting a Hedge Fund : A US Perspective,Book,NaN,1,0.0,1.386,1.386,Books[283155],Subjects[1000],Business & Investing[3],Investing[2665],2.333,20.333,17.333
402682,548543,0970020503,Facts Every Injured Worker Should Know,Book,NaN,2,0.0,1.792,1.792,Books[283155],Subjects[1000],Law[10777],Business[173486],4.400,1.400,1.400
402683,548544,B000065AHM,Lucky Man,Music,NaN,4,0.0,0.693,0.693,Music[5174],Specialty Stores[468040],Indie Music[266023],Blues[171241],5.000,2.000,2.000
402684,548545,B0000508ZN,I Need Your Loving,Music,NaN,3,0.0,0.693,0.693,Music[5174],Specialty Stores[468040],Indie Music[266023],Dance & DJ[171243],1.000,2.000,0.000
402685,548546,1930519206,Adobe Photoshop 6 VTC Training CD,Book,NaN,3,0.0,1.099,1.099,Books[283155],Subjects[1000],Computers & Internet[5],Graphics & Illustration[4134],5.000,1.000,0.500
402686,548547,B000059TOC,The Drifter,DVD,630366704X,14,0.0,0.693,0.693,[139452],DVD[130],Special Features[408328],Today's Deals in DVD[409298],5.000,2.000,0.000
402687,548548,B00006JBIX,The House Of Morecock,DVD,B00004WZQN,6,0.0,2.197,1.792,[139452],DVD[130],Specialty Stores[498862],Independently Distributed[901596],2.200,7.200,5.000
402688,548549,0879736836,Catholic Bioethics and the Gift of Human Life,Book,"1580510469,0896229939",3,0.0,0.693,0.693,Books[283155],Subjects[1000],Nonfiction[53],Social Sciences[11232],4.000,7.000,3.000
402689,548550,B00008DDST,"1, 2, 3 Soleils: Taha, Khaled, Faudel",DVD,NaN,3,0.0,1.386,1.386,[139452],DVD[130],Genres[404276],Music Video & Concerts[163420],5.000,2.000,2.000
402690,548551,B00005MHUG,That Travelin' Two-Beat/Sings the Great Countr...,Music,"B00005O6KL,B0000634HG",6,0.0,0.693,0.693,Music[5174],Styles[301668],Pop[37],Vocal Pop[406646],5.000,9.000,9.000


## Item embedding
Text embedding (sentence-BRET).

GNNs operate on a graph with each node having a feature vector. These feature vectors should:
1. Be fixed-length (not variable-length sequences)
2. Be dense (not sparse like TF-IDF)
3. Capture semantic meaning, especially for reviews


- if got errors like:Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with pip install tf-keras. run:




<pre> ```
1. pip install torch
2. pip uninstall keras tensorflow keras-nightly keras-preprocessing
3. pip install --upgrade sentence-transformers
``` </pre>

In [ ]:
# 1.1.1.Text embedding sentence-BRET (SBERT) for title and review, fixed lenghth, dense, capture semantic meaning
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
title_embeddings = model.encode(items_df['title'])
print(title_embeddings.shape)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(402691, 384)


In [ ]:
# 1.1.2. One-Hot Encode
from sklearn.preprocessing import OneHotEncoder, StandardScaler
group_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # dense encoder
group_encoded = group_encoder.fit_transform(items_df[['group']])
print(group_encoded.shape)
group_encoded[:2]

In [ ]:
# 1.1.3: Normalize numeric columns
# salesrank_log category_count reviews_total_log  reviews_downloaded_log  reviews_avg_ratings  reviews_avg_votes reviews_avg_helpful
numeric_cols = ['salesrank_log', 'category_count', 'reviews_total_log', 'reviews_downloaded_log', 'reviews_avg_ratings', 'reviews_avg_votes', 'reviews_avg_helpful']
items_df[numeric_cols] = items_df[numeric_cols].fillna(items_df[numeric_cols].mean())
scaler = StandardScaler()
numeric_features = scaler.fit_transform(items_df[numeric_cols])
print(numeric_features.shape)
numeric_features[:5]

## Category Embedding
***Option1: TFIDF***

***Option2: Label Encoding + Embedding***

***Option3: Graph based***

### Option1: TFIDF (obsolete/ not recommend this time)
ASIN + category_paths
1. TF-IDF embedding
2. maybe Word2Vec over category token?
TF-IDF stands for:
TF = Term Frequency
IDF = Inverse Document Frequency
Together, they form a statistic that measures how important a word is in a document, relative to a collection of documents (called a corpus).

Formula:
1) TF: How often the word appear in the document?
   - TF(w,d)= (total words in d)/ (count of w in d)

2) IDF: How rare is the word across all document?
    - IDF(w,D)=log( N/1+ nw)

   - N: total number of documents
   - nwL number of documents containg word w
3) TF-IDF(w,d,D) = TF(w,d) * IDF (w, D)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from datasets import Dataset, load_dataset

#### TF-IDF sparse Matrix

In [ ]:
# combine category paths into one string per product: (clean ver category 1-4)
category_cols = [f'category_path_{i}' for i in range(1, 5)]
items_df[category_cols] = categories_df[category_cols].fillna('') # no missing value. optional

In [ ]:
# Combine all category paths into a single space-separated string
items_df["category_str"] = items_df[category_cols].agg(' '.join, axis=1).str.strip()
items_ids = items_df['item_Id'].values
category_strs = items_df['category_str'].values


In [ ]:
# apply TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=1000)
tfidf_category = vectorizer.fit_transform(category_strs)  # CSR sparse matrix

# store item_Id for each row
category_tfidf_df = pd.DataFrame({
    'item_Id': items_df['item_Id'].values,
    'row_index': np.arange(tfidf_category.shape[0])
})

In [ ]:
print(category_tfidf_df.shape)

In [ ]:
print(category_tfidf_df[:5])

#### TF-IDF dense Matrix

In [ ]:
# truncatedSVD on sparce matrix, determine the optimal n_components. ref: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components = 300) # start at high n_components value
svd.fit(tfidf_category)


In [ ]:
cumulative = np.cumsum(svd.explained_variance_ratio_)
plt.plot(cumulative)
plt.xlabel("Components")
plt.ylabel("Cumulative Explained Variance")
plt.axhline(0.90, color='r', linestyle='--')
plt.axhline(0.85, color='b', linestyle='--')
plt.title("SVD Explained Variance")
plt.grid()
plt.show()


In [ ]:
# n_components= 200 captures ~80% variance, to keep small dimension while capture more features. can be adjusted later based on model
svd_200 = TruncatedSVD(n_components=200, random_state=42)
dense_category = svd_200.fit_transform(tfidf_category)  # output shape: (N, 200)
print(dense_category.shape)

In [ ]:
dense_category_df = pd.DataFrame(
    dense_category,
    columns=[f'cat_svd_{i}' for i in range(dense_category.shape[1])]
)
dense_category_df['item_Id'] = items_df['item_Id'].values
print(category_dense_df.shape)
print(category_dense_df.head(2))

#### Merge and combine features.

In [ ]:
print(title_embeddings.shape)
print(group_encoded.shape)
print(numeric_features.shape)
print(dense_category_df.shape) # it has item_id column

In [ ]:
X_svd = np.concatenate([
    title_embeddings,     # (N, 384)
    group_encoded,          # (N, 10)
    numeric_features,     # (N, 7)
    dense_category_df.drop(columns='item_Id').values,       # (N, 200)
], axis=1)

In [ ]:
X_svd.shape

In [ ]:
# column names
num_title = title_embeddings.shape[1]
num_group = group_encoded.shape[1]
num_numeric = numeric_features.shape[1]
num_category = dense_category_df.drop(columns='item_Id').shape[1] # drop item

column_names = (
    [f"title_{i}" for i in range(num_title)] +
    [f"group_{i}" for i in range(num_group)] +
    [f"num_{i}" for i in range(num_numeric)] +
    [f"category_{i}" for i in range(num_category)]
)
print(len(column_names))


In [ ]:
subfolder_path = os.path.join("data", "feature_embedding")
os.makedirs(subfolder_path, exist_ok=True)

In [ ]:
df_X_svd = pd.DataFrame(X_svd, columns=column_names)
np.save("data/feature_embedding/feature_matrix_svd.npy", df_X_svd)

In [ ]:
item_id = items_df['item_Id'].values
np.save("data/feature_embedding/item_id.npy", item_id)

### Option2 Label Encoding + Embedding
LabelEncoder + Embedding is trainable, task-aware, and semantically richer — especially for GNNs.

What is it?
- A method to convert categorical data (like category paths) into dense, trainable vector representations, optimized for a downstream task (e.g., GNN node classification, recommendation).
Key components:
| Step                       | What it Does                                                                                                                     |
| -------------------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| **1. Label Encoding**      | Converts each unique category/path to a unique integer ID. E.g., `"Books > Religion > Christianity"` → `ID = 4178`               |
| **2. Embedding Layer**     | A lookup table: maps each ID to a dense vector. E.g., `nn.Embedding(num_paths, dim)`                                             |
| **3. Task-Aware Learning** | The embedding vectors are **updated during training** (e.g., through GNN backpropagation) to capture **task-specific semantics** |

Character:
| Benefit                                | Explanation                                                                         |
| -------------------------------------- | ----------------------------------------------------------------------------------- |
| **Dense representation**             | Each category gets a compact vector (e.g., 64-dim)                                  |
| **Trainable and dynamic**            | Embeddings update during training to better support your task                       |
| **Learns semantic relationships**    | Similar categories will have closer vectors *if they behave similarly in your data* |
| **Efficient for large vocabularies** | Much smaller than one-hot or multi-hot vectors                                      |
| **Compatible with GNNs**             | Easily used as node features or pooled with other embeddings                        |


🚫 What it doesn't do by itself:

❌ Does not preserve hierarchy unless the model learns it from context

❌ The integer ID itself means nothing (no ordering or structure)- embedding process learned semantic relationships


In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder

In [ ]:
category_cols = [f'category_path_{i}' for i in range(1, 5)]
# Create a list of category path lists (per item)
category_paths_list = items_df[category_cols].values.tolist()
category_paths_list = [[p for p in paths if pd.notnull(p)] for paths in category_paths_list]

# Flatten all paths to get unique ones
all_paths = sorted(set(path for paths in category_paths_list for path in paths))

In [ ]:
# Label encode unique path strings
path_encoder = LabelEncoder()
path_encoder.fit(all_paths)

# Encode each item's list of paths
encoded_paths_list = [path_encoder.transform(paths) for paths in category_paths_list]

In [ ]:
# Create embedding layer
num_paths = len(path_encoder.classes_)
embed_dim = 64  # adjust as needed
category_embedding = nn.Embedding(num_paths, embed_dim)


In [ ]:
# Mean-pool embeddings for each item
def get_pooled_embedding(path_ids):
    path_ids_tensor = torch.tensor(path_ids, dtype=torch.long)
    emb = category_embedding(path_ids_tensor)  # shape: (num_paths_i, embed_dim)
    return emb.mean(dim=0)


In [ ]:
pooled_vectors = [get_pooled_embedding(encoded_paths) for encoded_paths in encoded_paths_list]
pooled_matrix = torch.stack(pooled_vectors)  # shape: (num_items, embed_dim)

category_feature_df = pd.DataFrame(
    pooled_matrix.detach().numpy(),
    columns=[f'cat_emb_{i}' for i in range(embed_dim)]
)
category_feature_df['item_Id'] = items_df['item_Id'].values

In [ ]:
# save to .npy file
np.save("data/feature_embedding/category_feature_option2.npy", category_feature_df)

### Option3: Graph-based embedding- Working in progress. not ideal
Adds semantics, good for GNN generalization, which explicitly turns the hierarchy structure into a learned embedding space.

***When to use:***

You have time and resources to build a small category graph.

You’re looking to push performance or explainability further.


***Why it's valuable:***

Embeds each category based on its position in the hierarchy.

Captures generalization like: "Christianity and Buddhism are both under Religion."

***What the embedding captures***
| Relationship in the hierarchy  | What happens in embedding space                        |
| ------------------------------ | ------------------------------------------------------ |
| Parent and child               | Vectors are **close** together                         |
| Siblings (under same parent)   | Vectors **cluster together**                           |
| Ancestors (multiple hops away) | Increasing **distance** between vectors                |
| Deep vs. shallow categories    | Embeddings can reflect **depth** or **specialization** |

This turns the tree/DAG structure into a continuous vector space where:

Semantic relationships are preserved

Similarity and hierarchical distance are measurable (e.g., via cosine or Euclidean distance)



In [ ]:
import pandas as pd
import networkx as nx
import re

In [ ]:
# Step1. Build Hierachical Graph
category_cols = [f'category_path_{i}' for i in range(1, 5)]
# Create a directed graph
G = nx.DiGraph()

# Extract parent-child edges from each path
for _, row in items_df[category_cols].iterrows():
    # Drop any missing levels (NaNs)
    path = [str(cat).strip() for cat in row if pd.notnull(cat)]
    # Add edges between consecutive levels
    for i in range(len(path) - 1):
        parent = path[i]
        child = path[i + 1]
        G.add_edge(parent, child)


In [ ]:
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Sample edges:", list(G.edges)[:10])

In [ ]:
# Step2 Train Node2Vec on the category graph # parameters may need further tuning
from node2vec import Node2Vec
node2vec = Node2Vec(G,
                    dimensions=64, # Output embedding size
                    walk_length=20, # Length of each random walk
                    num_walks=50, # Number of walks per node
                    p=1, # return parameter
                    q=0.25,  # In-out (exploration) parameter
                    workers=2) # default initialization
model = node2vec.fit(window=10, # Context window for Word2Vec
                     min_count=1, # Include all nodes in training
                     batch_words=4)

How p and q  affect the walk?

Imagine a walk just visited node t → and is now at node v.
Now it’s choosing where to go next (from v to some neighbor x).

The decision is weighted based on:
If x == t (going back to where it came from): influenced by p

If x is a neighbor of t (close): controlled by q

If x is not close to t (far): also controlled by q
| Goal                                  | Setting            |
| ------------------------------------- | ------------------ |
| Learn hierarchical/semantic structure | `p = 1`, `q = 0.5` |
| Emphasize local communities           | `p = 1`, `q = 2`   |
| Balanced exploration                  | `p = 1`, `q = 1`   |


In [ ]:
# Step 3: Extract category embeddings from the Node2Vec model
category_embeddings = {
    node: model.wv[node] for node in G.nodes if node in model.wv
}
embedding_dim = model.vector_size

In [ ]:
import seaborn as sns

In [ ]:
# 1. Build category -> top-level path mapping from items_df
category_to_root = {}

for _, row in items_df.iterrows():
    top = row['category_path_1']
    for col in ['category_path_1', 'category_path_2', 'category_path_3', 'category_path_4']:
        cat = row[col]
        if pd.notnull(cat):
            category_to_root[cat] = top

# 2. Assign top-level labels to each node2vec node
labels = list(model.wv.index_to_key)
top_level_labels = [category_to_root.get(label, 'Unknown') for label in labels]


In [ ]:
len(top_level_labels)

In [ ]:
from collections import Counter
# Count frequency of each top-level category
top_level_counter = Counter(top_level_labels)

# Print how many unique top-level categories
print(f"Number of unique top-level categories: {len(top_level_counter)}")

# Show top 10 most common categories
print("Top-level category distribution:")
for category, count in top_level_counter.most_common(10):
    print(f"{category}: {count}")

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=top_level_labels, palette='tab20', alpha=0.7)
plt.title("t-SNE of Category Embeddings Colored by Top-Level Category")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.grid(True)
plt.show()

In [ ]:
# Step4: Pool category embeddings for each item
import numpy as np

def get_item_category_embedding(row):
    embeddings = []
    for col in ['category_path_1', 'category_path_2', 'category_path_3', 'category_path_4']:
        cat = row.get(col)
        if pd.notnull(cat) and cat in category_embeddings:
            embeddings.append(category_embeddings[cat])
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(embedding_dim)

# Apply to each row
items_df['category_embedding'] = items_df.apply(get_item_category_embedding, axis=1)

# Convert to matrix
cat_emb_matrix = np.vstack(items_df['category_embedding'].values)
cat_emb_df = pd.DataFrame(cat_emb_matrix, columns=[f'cat_emb_{i}' for i in range(embedding_dim)])
cat_emb_df['item_Id'] = items_df['item_Id'].values

## (Optional)  Title embedding pca analysis

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# apply pca
pca= PCA ()
title_pca = pca.fit_transform(title_embeddings)
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(cumulative_variance, marker='o')
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA - Cumulative Explained Variance for Title Embeddings")
plt.grid(True)
plt.axhline(0.95, color='r', linestyle='--', label='95% variance')
plt.axhline(0.90, color='g', linestyle='--', label='90% variance')
plt.legend()
plt.show()


90% variance arond 200 components

In [ ]:
pca_200 = PCA(n_components = 200)
title_embeddings_reduced = pca_200.fit_transform(title_embeddings)

In [ ]:
X_pca = np.concatenate([
    title_embeddings_reduced,     # (N, 200)
    group_encoded,          # (N, 10)
    numeric_features,     # (N, 7)
    # dense_category_df.drop(columns='item_Id').values,       # (N, 200) (Or concate your own category features)
], axis=1)

In [ ]:
print(X_pca.shape)

In [ ]:
# column names
num_title = title_embeddings_reduced.shape[1]
num_group = group_encoded.shape[1]
num_numeric = numeric_features.shape[1]
num_category = dense_category_df.drop(columns='item_Id').shape[1] # drop item

column_names = (
    [f"title_{i}" for i in range(num_title)] +
    [f"group_{i}" for i in range(num_group)] +
    [f"num_{i}" for i in range(num_numeric)]
    # [f"category_{i}" for i in range(num_category)]
)
print(len(column_names))

df_X_pca = pd.DataFrame(X_pca, columns=column_names)

In [ ]:
# save to .npy file
np.save("data/feature_embedding/features_matrix_title_pca_nocat.npy", df_X_pca)

In [ ]:
# np.savez_compressed
np.savez_compressed("data/feature_embedding/features_matrix_title_pca_nocat.npy", df_X_pca)

In [ ]:
X = np.concatenate([
    title_embeddings,     # (N, 200)
    group_encoded,          # (N, 10)
    numeric_features,     # (N, 7)
    # dense_category_df.drop(columns='item_Id').values,       # (N, 200) (Or concate your own category features)
], axis=1)

In [ ]:
np.savez_compressed("data/feature_embedding/feature_matrix_no_cat.npy", X)

In [ ]:
np.save("data/feature_embedding/feature_matrix_no_cat.npy", X)

# References:
1. NetworkX: https://networkx.org/documentation/stable/auto_examples/index.html
2. pytorch GCN github: https://github.com/tkipf/pygcn
3. GraphSage github: https://github.com/williamleif/graphsage-simple